# Prepare the real-world datasets

Downloads TechQA (IBM), Stack Exchange (Super User, Ask Ubuntu) and the UCI ServiceNow incident log, converts them into the project's tables and prints checks. No GPU needed.

In [ ]:
REPO_URL = 'https://github.com/rushitpatel100-oss/ticket-triage-ai.git'
import os
if not os.path.exists('/content/ticket-triage-ai'):
    !git clone -q $REPO_URL /content/ticket-triage-ai
%cd /content/ticket-triage-ai
!git pull -q
!git log --oneline -1

In [ ]:
!python -m src.sources

In [ ]:
import json, zipfile
import pandas as pd
from src import config
from src.sources import common, itsm_log

checks = {}

# TechQA: raw file layout and how many gold documents the corpus already contains
raw = config.RAW_DIR / "techqa"
with zipfile.ZipFile(raw / "corpus.zip") as zf:
    names = [n for n in zf.namelist() if not n.endswith("/")]
docs, q = common.load_tables("techqa")
corpus_ids = {"techqa:" + n.rsplit("/", 1)[-1].rsplit(".", 1)[0] for n in names if n.endswith(".txt")}
gold = {g for gs in q["gold_doc_ids"] for g in gs}
checks["techqa"] = {
    "zip_members": len(names), "zip_examples": names[:3],
    "queries": len(q), "impossible": int((~q["answerable"]).sum()),
    "gold_docs": len(gold), "gold_docs_in_corpus": len(gold & corpus_ids),
    "split_x_answerable": {f"{s}/{a}": int(n) for (s, a), n in q.groupby(["split", "answerable"]).size().items()},
    "query_example": q["text"].iloc[0][:200], "doc_example": docs["text"].iloc[0][:200],
    "id_examples": q["query_id"].head(2).tolist() + q["query_id"].tail(2).tolist(),
}

# Stack Exchange: what the raw question/answer text looks like before cleaning
files = sorted((config.RAW_DIR / "stackexchange" / "data" / "superuser.com").glob("*.parquet"))
raw_se = pd.read_parquet(files[0])
docs, q = common.load_tables("stackexchange")
checks["stackexchange"] = {
    "raw_columns": list(raw_se.columns), "raw_rows_first_file": len(raw_se),
    "raw_question_example": str(raw_se["question"].iloc[0])[:300],
    "raw_answer_keys": sorted(raw_se["answers"].iloc[0][0].keys()),
    "raw_selected_type": type(raw_se["answers"].iloc[0][0]["selected"]).__name__,
    "queries": len(q), "docs": len(docs),
    "clean_question_example": q["text"].iloc[0][:200], "clean_answer_example": docs["text"].iloc[0][:200],
}

# UCI log: check the formats assumed by the loader
ev = pd.read_csv(config.RAW_DIR / "itsm_log" / itsm_log.ZIP_NAME, na_values=["?"], low_memory=False,
                 compression="zip")
parsed = itsm_log.normalise(ev)
checks["itsm_log"] = {
    "raw_columns": list(ev.columns),
    "raw_examples": {c: ev[c].dropna().astype(str).unique()[:6].tolist() for c in
                     ["incident_state", "opened_at", "resolved_at", "priority", "made_sla", "knowledge", "contact_type"]
                     if c in ev},
    "unparsed_dates": {c: int(parsed[c].isna().sum() - ev[c].isna().sum()) for c in itsm_log.DATE_COLUMNS if c in ev},
    "incidents_without_resolution_hours": int((pd.read_parquet(config.DATA_DIR / "itsm_log" / "incidents.parquet")["resolution_hours"].isna()).sum()),
}
print("CHECKS_START")
print(json.dumps(checks, indent=1, default=str))
print("CHECKS_END")

In [ ]:
print(open('results/real_data_summary.json').read())